# Aktywacja Strategii Odbicie Ogórkowe
Ten notatnik sĹ‚uĹĽy do testowania, wizualizacji i optymalizacji strategii powrotu do Ĺ›redniej po mocnych spadkach.

## Importy, dane i sygnały

In [1]:
import os
import sys

# Dodajemy folder glowny do path aby moduly dzialaly
sys.path.append(os.path.abspath('c:/Users/PC/Documents/Antigravity/lyse-lby'))
os.chdir('c:/Users/PC/Documents/Antigravity/lyse-lby')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

# Importujemy ladowanie danych i nasze nowe moduly
from core.ladowanie_danych import create_stock_dfs
from odbicie.mackowe_sygnaly import mackowe_sygnaly
from odbicie.odbicie import generate_odbicie_entries
from odbicie.odbicie_atr import generate_odbicie_atr_entries
from odbicie.odbicie_bb import generate_odbicie_bb_entries
from odbicie.tbm import moving_triple_barrier_labels
from odbicie.optymalizacja import optimize_atr_tbm, optimize_bb_tbm

# Katalog na pliki cache - zawsze wewnatrz odbicie/dane/, niezalezny od cwd
# __vsc_ipynb_file__ dostepny w VS Code; fallback na abspath('')
_SCRIPT_DIR = os.path.dirname(os.path.abspath(globals()["__vsc_ipynb_file__"])) if "__vsc_ipynb_file__" in globals() else os.path.abspath("")
# Jesli notebook lezy wewnatrz odbicie/, katalog jest katalogiem rodzica
if os.path.basename(_SCRIPT_DIR) != 'odbicie':
    _SCRIPT_DIR = os.path.join(_SCRIPT_DIR, 'odbicie')
DANE_DIR = os.path.join(_SCRIPT_DIR, 'dane')
os.makedirs(DANE_DIR, exist_ok=True)
print(f"Cache dir: {DANE_DIR}")

# Ustawienia ladowania danych
import json
with open(os.path.join(_SCRIPT_DIR, 'settings.json'), 'r') as f:
    all_settings = json.load(f)

# Ustawienia ladowania danych
settings = all_settings['data_settings']

Cache dir: c:\Users\PC\Documents\Antigravity\lyse-lby\odbicie\dane


In [2]:
# 1. Ladowanie Danych
import pickle

data_cache_file = os.path.join(DANE_DIR, 'dfs_cache.pkl')

if os.path.exists(data_cache_file):
    print("Znaleziono zapisane dane. Wczytywanie z pliku...")
    with open(data_cache_file, "rb") as f:
        dfs_1d, dfs_1w = pickle.load(f)
    print(f"Wczytano {len(dfs_1d)} symboli 1D i {len(dfs_1w)} symboli 1W z pliku {data_cache_file}.")
else:
    print("Ladowanie danych dziennych i tygodniowych...")
    dfs_1d, dfs_1w = create_stock_dfs(settings)
    print(f"Zaladowano {len(dfs_1d)} symboli 1D i {len(dfs_1w)} symboli 1W.")
    print("Zapisywanie danych do pliku...")
    with open(data_cache_file, "wb") as f:
        pickle.dump((dfs_1d, dfs_1w), f)
    print("Dane zapisane pomyslnie.")


Znaleziono zapisane dane. Wczytywanie z pliku...
Wczytano 478 symboli 1D i 478 symboli 1W z pliku c:\Users\PC\Documents\Antigravity\lyse-lby\odbicie\dane\dfs_cache.pkl.


In [3]:
# 2. Generowanie Sygnalow Bazowych (mackowe_sygnaly)
signals_cache_file = os.path.join(DANE_DIR, 'signals_cache.pkl')

if os.path.exists(signals_cache_file):
    print("Znaleziono zapisane sygnaly. Wczytywanie z pliku...")
    with open(signals_cache_file, "rb") as f:
        signals_df = pickle.load(f)
    print(f"Wczytano {len(signals_df)} sygnalow z pliku {signals_cache_file}.")
else:
    signals_df = mackowe_sygnaly(
        dfs=dfs_1w,
        settings=settings,
        require_vol_confirmation=True,
        require_cmo_confirmation=True,
        interval='1w',
        entry_offset=0,
        pattern_cols=['hammer', 'inverted_hammer', 'engulfing_bull', 'piercing_line'],
        debug=True
    )
    print("Zapisywanie sygnalow do pliku...")
    with open(signals_cache_file, "wb") as f:
        pickle.dump(signals_df, f)
    print("Sygnaly zapisane pomyslnie.")

signals_df.describe()


Znaleziono zapisane sygnaly. Wczytywanie z pliku...
Wczytano 912 sygnalow z pliku c:\Users\PC\Documents\Antigravity\lyse-lby\odbicie\dane\signals_cache.pkl.


,signal_time,entry_time,signal_close
count,912,912,912.000000
mean,2023-10-08 09:01:34.736842,2023-10-08 09:01:34.736842,129.103488
min,2021-07-25 00:00:00,2021-07-25 00:00:00,7.970000
25%,2022-05-15 00:00:00,2022-05-15 00:00:00,51.740002
50%,2023-10-08 00:00:00,2023-10-08 00:00:00,97.340000
75%,2025-03-16 00:00:00,2025-03-16 00:00:00,179.127495
max,2026-02-22 00:00:00,2026-02-22 00:00:00,565.369995
std,NaN,NaN,103.256636


## Wejście i Wyjście

In [4]:
# 3. Wybor Strategii Wejscia
# Zmien ta zmienna aby przełaczyc strategie: 'base', 'atr', 'bb'
STRATEGY = 'bb'

strat_settings = all_settings['strategies'][STRATEGY]['entry_settings']
tbm_settings = all_settings['strategies'][STRATEGY]['tbm_settings']

if STRATEGY == 'base':
    # --- Strategia bazowa: staly prog procentowy ---
    threshold_pct = strat_settings['threshold_pct']
    max_hold = strat_settings['max_setup_hold_bars']
    entries_df = generate_odbicie_entries(
        signals_df=signals_df,
        market_data_daily=dfs_1d,
        threshold_pct=threshold_pct,
        max_setup_hold_bars=max_hold
    )
    print(f"[base] Wygenerowano {len(entries_df)} wejsc przy progu {threshold_pct*100}%")

elif STRATEGY == 'atr':
    # --- Strategia ATR: prog oparty na wielokrotnosci ATR ---
    atr_period = strat_settings['atr_period']
    atr_factor = strat_settings['atr_factor']
    max_hold = strat_settings['max_setup_hold_bars']
    entries_df = generate_odbicie_atr_entries(
        signals_df=signals_df,
        market_data_daily=dfs_1d,
        atr_period=atr_period,
        atr_factor=atr_factor,
        max_setup_hold_bars=max_hold
    )
    print(f"[atr] Wygenerowano {len(entries_df)} wejsc (period={atr_period}, factor={atr_factor})")

elif STRATEGY == 'bb':
    # --- Strategia BB: wejscie przy dotknięciu dolnej wstegi Bollingera ---
    bb_period = strat_settings['bb_period']
    bb_std   = strat_settings['bb_std']
    max_hold = strat_settings['max_setup_hold_bars']
    entries_df = generate_odbicie_bb_entries(
        signals_df=signals_df,
        market_data_daily=dfs_1d,
        bb_period=bb_period,
        bb_std=bb_std,
        max_setup_hold_bars=max_hold
    )
    print(f"[bb] Wygenerowano {len(entries_df)} wejsc (period={bb_period}, std={bb_std})")

else:
    raise ValueError(f"Nieznana strategia: '{STRATEGY}'. Uzyj 'base', 'atr' lub 'bb'.")

entries_df.head()


[bb] Wygenerowano 80 wejsc (period=10, std=3.0)


,symbol,signal_time,pattern,entry_time,entry_price,signal_close,bb_period,bb_std,bb_lower,bb_middle,bb_upper,bb_bandwidth,rsi_at_entry,entry_atr,setup_bars
0,AOS,2025-01-19,engulfing_bull,2025-01-30,66.308652,71.809998,10,3.0,66.308652,70.654000,74.999348,0.123004,37.050777,1.620781,8
1,ACN,2024-05-05,hammer,2024-05-17,300.827487,303.709991,10,3.0,300.827487,307.611996,314.396506,0.044111,35.487476,5.746611,10
2,ACN,2024-06-23,engulfing_bull,2024-07-08,295.711708,308.980011,10,3.0,295.711708,303.634000,311.556291,0.052183,49.336508,6.465444,10
3,AEE,2023-10-15,engulfing_bull,2023-10-23,74.008169,77.949997,10,3.0,74.008169,77.250000,80.491831,0.083931,41.028454,1.673912,6
4,AEP,2024-12-22,hammer,2025-01-06,90.112525,92.750000,10,3.0,90.112525,92.115001,94.117477,0.043478,32.078326,1.469593,9


In [5]:
# 4. Wyjście z użyciem Moving Triple Barrier Method
trades_df = moving_triple_barrier_labels(
    entries_df=entries_df,
    market_data_daily=dfs_1d,
    tp_mult=tbm_settings['tpm'],
    sl_mult=tbm_settings['slm'],
    tp_trail_mult=tbm_settings['ttpm'],
    max_holding_bars=tbm_settings['mhb'],
    early_breakeven=tbm_settings['early_bailout'],
    time_decay_sl=tbm_settings['time_decay_sl'],
    active_trailing_sl=tbm_settings['active_trail_sl'],
    sl_trail_mult=tbm_settings['sl_trail_mult'],
    max_loss_pct=tbm_settings['max_loss_pct'],
    exit_on_close=tbm_settings.get('exit_on_close', True)
)
print(f"Zakończono {len(trades_df)} transakcji.")
trades_df.head()


Zakończono 80 transakcji.


,symbol,signal_time,pattern,entry_time,entry_price,signal_close,bb_period,bb_std,bb_lower,bb_middle,bb_upper,bb_bandwidth,rsi_at_entry,entry_atr,setup_bars,exit_time,exit_price,return_pct,exit_reason,hold_bars
0,AOS,2025-01-19,engulfing_bull,2025-01-30,66.308652,71.809998,10,3.0,66.308652,70.654000,74.999348,0.123004,37.050777,1.620781,8,2025-02-03,67.040001,1.102946,TRAILING_TP,2
1,ACN,2024-05-05,hammer,2024-05-17,300.827487,303.709991,10,3.0,300.827487,307.611996,314.396506,0.044111,35.487476,5.746611,10,2024-05-21,303.640015,0.934930,TRAILING_TP,2
2,ACN,2024-06-23,engulfing_bull,2024-07-08,295.711708,308.980011,10,3.0,295.711708,303.634000,311.556291,0.052183,49.336508,6.465444,10,2024-07-10,295.440002,-0.091882,TRAILING_TP,2
3,AEE,2023-10-15,engulfing_bull,2023-10-23,74.008169,77.949997,10,3.0,74.008169,77.250000,80.491831,0.083931,41.028454,1.673912,6,2023-10-27,75.379997,1.853617,TRAILING_TP,4
4,AEP,2024-12-22,hammer,2025-01-06,90.112525,92.750000,10,3.0,90.112525,92.115001,94.117477,0.043478,32.078326,1.469593,9,2025-01-13,94.540001,4.913275,TRAILING_TP,4


## Analiza

In [6]:
# 5. Analiza i Statystyki
if not trades_df.empty:
    wins = (trades_df['return_pct'] > 0).sum()
    losses = (trades_df['return_pct'] <= 0).sum()
    win_rate = wins / len(trades_df) * 100
    trades_df['return_per_bar'] = trades_df['return_pct'] / trades_df['hold_bars']
    
    print(f"Total Trades: {len(trades_df)}")
    print(f"Win Rate: {win_rate:.2f}%")
    print(f"Avg Return: {trades_df['return_pct'].mean():.2f}%")
    print(f"Avg bars held: {trades_df['hold_bars'].mean():.2f}")
    print(f"Avg Return per Bar: {trades_df['return_per_bar'].mean():.2f}%")

    
    # Powody wyjĹ›cia
    print("\nExit Reasons:")
    print(trades_df['exit_reason'].value_counts())
else:
    print("Brak transakcji do analizy.")

Total Trades: 80
Win Rate: 78.75%
Avg Return: 3.73%
Avg bars held: 2.71
Avg Return per Bar: 1.30%

Exit Reasons:
exit_reason
TRAILING_TP    77
TIME_EXIT       2
SL              1
Name: count, dtype: int64


In [7]:
temp = pd.DataFrame({
    'count': trades_df.groupby('exit_reason').return_pct.count(),
    'avg_return': trades_df.groupby('exit_reason').return_pct.mean(),
    'cumulativ_return': trades_df.groupby('exit_reason').return_pct.sum(),
    'std': trades_df.groupby('exit_reason').return_pct.std(),
    'avg_hold_bars': trades_df.groupby('exit_reason').hold_bars.mean(),
    'std_hold_bars': trades_df.groupby('exit_reason').hold_bars.std(),
    'max_hold_bars': trades_df.groupby('exit_reason').hold_bars.max()
    })
    
temp

,count,avg_return,cumulativ_return,std,avg_hold_bars,std_hold_bars,max_hold_bars
exit_reason,,,,,,,
SL,1,-7.716723,-7.716723,NaN,6.000000,NaN,6
TIME_EXIT,2,6.240760,12.481521,14.678155,7.000000,0.000000,7
TRAILING_TP,77,3.813992,293.677391,6.287493,2.558442,0.910372,7


## Ploty

In [8]:
# 6. Interaktywna Wizualizacja Transakcji
from odbicie.plot import show_trade_viewer
import ipywidgets as widgets
from IPython.display import display, clear_output

# Inicjalny rysunek
show_trade_viewer(
    trades_df,
    dfs_1d,
    tpm=tbm_settings['tpm'],
    slm=tbm_settings['slm'],
    ttpm=tbm_settings['ttpm'],
    mhb=tbm_settings['mhb'],
    exit_reason='All',
    active_trailing_sl=tbm_settings['active_trail_sl'],
    sl_trail_mult=tbm_settings['sl_trail_mult'],
    max_loss_pct=tbm_settings['max_loss_pct'],
    time_decay_sl=tbm_settings['time_decay_sl'],
    exit_on_close=tbm_settings.get('exit_on_close', True),
    strategy_type="tbm"
)


Output()

## Optymalizacja

### Stare

In [9]:
# Optymalizacja Progu WejĹ›cia i Czasu Trzymania Setupu
import itertools
from tqdm.notebook import tqdm

def optimize_threshold(thresholds, max_holding_bars):
    results = []
    
    # Tworzymy siatkÄ™ wszystkich kombinacji wejĹ›ciowych list
    grid = list(itertools.product(thresholds, max_holding_bars))
    
    for th, max_bars in tqdm(grid, desc="Optymalizacja progu"):
        # max_bars definiuje ile dni po sygnale czekamy na wpadniÄ™cie w prĂłg
        ents = generate_odbicie_entries(signals_df, dfs_1d, threshold_pct=th, max_setup_hold_bars=max_bars)
        
        # max_bars definiuje rĂłwnieĹĽ jak dĹ‚ugo trzymamy trade zanim zamkniemy na czas
        trds = moving_triple_barrier_labels(ents, dfs_1d, tp_mult=tpm, sl_mult=slm, tp_trail_mult=ttpm, max_holding_bars=max_bars)
        
        if len(trds) > 0:
            win_rate = (trds['return_pct'] > 0).mean() * 100
            avg_return = trds['return_pct'].mean()
            results.append({
                'threshold_pct': th,
                'max_holding_bars': max_bars,
                'trades': len(trds),
                'win_rate': win_rate,
                'avg_return': avg_return
            })
            
    df_res = pd.DataFrame(results)
    if not df_res.empty:
        df_res = df_res.sort_values(by='avg_return', ascending=False)
    return df_res

'''
thresholds = [9,10,11,12]
max_holding_bars = [15]

print("Uruchamianie optymalizacji progu...")
opt_df = optimize_threshold(thresholds, max_holding_bars)

display(opt_df.head(10))
'''


'\nthresholds = [9,10,11,12]\nmax_holding_bars = [15]\n\nprint("Uruchamianie optymalizacji progu...")\nopt_df = optimize_threshold(thresholds, max_holding_bars)\n\ndisplay(opt_df.head(10))\n'

In [10]:
# Optymalizacja ParametrĂłw TBM (Take Profit / Stop Loss / Max Hold)
import itertools
from tqdm.notebook import tqdm
import pandas as pd

def optimize_tbm(entries_df, market_data_daily, tp_mults, sl_mults, trail_activations, max_holding_bars_list):
    results = []
    
    # Tworzymy siatkÄ™ wszystkich kombinacji
    grid = list(itertools.product(tp_mults, sl_mults, trail_activations, max_holding_bars_list))
    
    for tp, sl, trail, max_bars in tqdm(grid, desc="Optymalizacja TBM"):
        trds = moving_triple_barrier_labels(
            entries_df=entries_df, 
            market_data_daily=market_data_daily, 
            tp_mult=tp, 
            sl_mult=sl, 
            tp_trail_mult=trail, 
            max_holding_bars=max_bars,
            early_breakeven=early_bailout,
            time_decay_sl=time_decay_sl,
            active_trailing_sl=active_trail_sl,
            sl_trail_mult=sl_trail_mult,
            max_loss_pct=max_loss_pct
        )
        
        if len(trds) > 0:
            win_rate = (trds['return_pct'] > 0).mean() * 100
            avg_return = trds['return_pct'].mean()
            avg_hold_bars = trds['hold_bars'].mean()

            results.append({
                'tp_mult': tp,
                'sl_mult': sl,
                'trail_activation': trail,
                'max_holding_bars': max_bars,
                'trades': len(trds),
                'win_rate': win_rate,
                'avg_return': avg_return,
                'avg_hold_bars': avg_hold_bars,
                'return_per_bar': avg_return / avg_hold_bars if avg_hold_bars > 0 else 0
            })

    df_res = pd.DataFrame(results)
    if not df_res.empty:
        df_res = df_res.sort_values(by='return_per_bar', ascending=False)
    return df_res


'''
tp_mults = [a/100 for a in range(113, 117, 1)]
sl_mults = [a/100 for a in range(313, 321, 1)]
trail_activations = [a/100 for a in range(1, 3, 1)]
max_holding_bars = [15]

print("Uruchamianie optymalizacji TBM. To moĹĽe zajÄ…Ä‡ chwilÄ™...")
tbm_opt_results = optimize_tbm(entries_df, dfs_1d, tp_mults, sl_mults, trail_activations, max_holding_bars)

display(tbm_opt_results.head(10))
'''


'\ntp_mults = [a/100 for a in range(113, 117, 1)]\nsl_mults = [a/100 for a in range(313, 321, 1)]\ntrail_activations = [a/100 for a in range(1, 3, 1)]\nmax_holding_bars = [15]\n\nprint("Uruchamianie optymalizacji TBM. To moĹĽe zajÄ…Ä‡ chwilÄ™...")\ntbm_opt_results = optimize_tbm(entries_df, dfs_1d, tp_mults, sl_mults, trail_activations, max_holding_bars)\n\ndisplay(tbm_opt_results.head(10))\n'

### ATR + TBM

In [11]:
# Optymalizacja wejsc ATR + wyjsc TBM
# Odkomentuj blok ponizej, aby uruchomic optymalizacje.


atr_tbm_settings = all_settings['strategies']['atr']['tbm_settings']
atr_opt_results = optimize_atr_tbm(
    signals_df=signals_df,
    market_data_daily=dfs_1d,
    atr_periods=[15],
    atr_factors=[4.0],
    max_setup_hold_bars_list=[10],
    tp_mults=[0.8],
    sl_mults=[1.75],
    tp_trail_mults=[0.07],
    max_holding_bars_list=[7],
    early_breakeven=atr_tbm_settings['early_bailout'],
    time_decay_sl=atr_tbm_settings['time_decay_sl'],
    active_trailing_sl=atr_tbm_settings['active_trail_sl'],
    sl_trail_mult=atr_tbm_settings['sl_trail_mult'],
    max_loss_pct=atr_tbm_settings['max_loss_pct'],
    exit_on_close=atr_tbm_settings.get('exit_on_close', True),
    min_trades=10
)
display(atr_opt_results.head(20))



Optymalizacja ATR + TBM:   0%|          | 0/1 [00:00<?, ?it/s]

,atr_period,atr_factor,max_setup_bars,buy_on_close,tp_mult,sl_mult,tp_trail_mult,max_holding_bars,trades,win_rate,avg_return,avg_hold_bars,avg_setup_bars,return_per_bar
0,15,4.0,10,False,0.8,1.75,0.07,7,40,65.0,3.72414,3.7,7.475,0.677861


### BB + TBM

In [12]:
# Optymalizacja wejsc BB + wyjsc TBM
# Odkomentuj blok ponizej, aby uruchomic optymalizacje.


bb_tbm_settings = all_settings['strategies']['bb']['tbm_settings']
bb_opt_results = optimize_bb_tbm(
    signals_df=signals_df,
    market_data_daily=dfs_1d,
    bb_periods=[10],
    bb_stds=[3.0],
    max_setup_hold_bars_list=[10],
    tp_mults=[0.2],
    sl_mults=[2.0],
    tp_trail_mults=[0.04],
    max_holding_bars_list=[7],
    early_breakeven=bb_tbm_settings['early_bailout'],
    time_decay_sl=bb_tbm_settings['time_decay_sl'],
    active_trailing_sl=bb_tbm_settings['active_trail_sl'],
    sl_trail_mult=bb_tbm_settings['sl_trail_mult'],
    max_loss_pct=bb_tbm_settings['max_loss_pct'],
    exit_on_close=bb_tbm_settings.get('exit_on_close', True),
    min_trades=10
)
display(bb_opt_results.head(20))



Optymalizacja BB + TBM:   0%|          | 0/1 [00:00<?, ?it/s]

,bb_period,bb_std,max_setup_bars,buy_on_close,tp_mult,sl_mult,tp_trail_mult,max_holding_bars,trades,win_rate,avg_return,avg_hold_bars,avg_setup_bars,return_per_bar
0,10,3.0,10,False,0.2,2.0,0.04,7,80,78.75,3.730527,2.7125,5.725,1.295317
